In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.stats import linregress

In [ ]:
data_dir = Path()
bio_path = Path()
grid_path = Path()
output_path = Path()

In [ ]:
merged_df = pd.concat(
    (pd.read_csv(f) for f in data_dir.glob("*.csv")),
    ignore_index=True
)

In [ ]:
pat = re.compile(r"\{[^}]*\}")

def parse_groups(s):
    rows = []
    for e in pat.findall(str(s)):
        ck = re.search(r"composite_key\s*=\s*([-+eE0-9\.]+)", e)
        m = re.search(r"mean\s*=\s*([-+eE0-9\.]+)", e)
        c = re.search(r"count\s*=\s*(\d+)", e)
        if ck and m and c:
            rows.append({
                "composite_key": float(ck.group(1)),
                "mean": float(m.group(1)),
                "count": int(c.group(1))
            })
    return rows

In [ ]:
long_df = (
    merged_df[["GRID_ID", "groups"]]
    .assign(parsed=lambda x: x["groups"].apply(parse_groups))
    .explode("parsed")
    .reset_index(drop=True)
)

parsed = pd.json_normalize(long_df["parsed"])
long_df = pd.concat([long_df.drop(columns=["groups", "parsed"]), parsed], axis=1)
long_df = long_df.dropna(subset=["composite_key"]).copy()

In [ ]:
BASE1 = 10**6
BASE2 = 10**3

ck = long_df["composite_key"].astype(int)

long_df["elev_bin"] = ck // BASE1
long_df["slope_bin"] = (ck % BASE1) // BASE2
long_df["aspect_bin"] = ck % BASE2

long_df["elev_center"] = long_df["elev_bin"] * 100 + 50

slope_map = {
    0: 2.5,
    1: 10,
    2: 20,
    3: 30,
    4: 40,
    5: 50,
    6: 60
}
long_df["slope_center"] = long_df["slope_bin"].map(slope_map)

long_df["aspect_dist_center"] = np.where(
    long_df["aspect_bin"] == 18,
    180,
    long_df["aspect_bin"] * 10 + 5
)

## observed AEI

In [ ]:
df = long_df[(long_df["mean"] > 0) & (long_df["count"] > 100)].copy()

df["aspect_cos"] = np.cos(np.deg2rad(df["aspect_dist_center"]))
df["log_mean"] = np.log(df["mean"])

df = df.groupby(["GRID_ID", "elev_bin", "slope_bin"]).filter(
    lambda g: g["aspect_bin"].nunique() >= 6
)

In [ ]:
def rmax_from_ab(a, b):
    if abs(b) < 1e-12:
        return 0.0

    disc = np.sqrt(1 + 4 * b * b)
    u1 = (-1 + disc) / (2 * b)
    u2 = (-1 - disc) / (2 * b)

    cand = [u for u in (u1, u2) if -1 <= u <= 1]
    if not cand:
        return 0.0

    best = None
    for u in cand:
        theta = np.arccos(u)
        r = (np.pi * a * b / 180) * np.sin(theta) * np.exp(b * u)
        if best is None or abs(r) > abs(best):
            best = r
    return float(best)


def fit_group(g):
    r = linregress(g["aspect_cos"], g["log_mean"])
    if r.pvalue >= 0.001:
        return None

    a = float(np.exp(r.intercept))
    b = float(r.slope)

    return pd.Series({
        "a": a,
        "b": b,
        "r2": float(r.rvalue ** 2),
        "p": float(r.pvalue),
        "rmax": rmax_from_ab(a, b)
    })

In [ ]:
fit_tbl = (
    df.groupby(["GRID_ID", "elev_center", "slope_center"])
      .apply(fit_group)
      .dropna()
      .reset_index()
)

In [ ]:
df_r = long_df.merge(
    fit_tbl,
    on=["GRID_ID", "elev_center", "slope_center"],
    how="left"
)

r_tbl = (
    df_r.dropna(subset=["rmax"])
        .groupby(["GRID_ID", "elev_center"])
        .agg(
            rmax=("rmax", "mean"),
            slope_center=("slope_center", "mean")
        )
        .reset_index()
)

## prepare data for XGB model

In [ ]:
bio = pd.read_csv(bio_path)

bio_wide = (
    bio.pivot_table(
        index=["GRID_ID", "elevation_bin_center"],
        columns="bio_variable",
        values="bio_mean"
    )
    .reset_index()
)

model_df = r_tbl.merge(
    bio_wide,
    left_on=["GRID_ID", "elev_center"],
    right_on=["GRID_ID", "elevation_bin_center"],
    how="left"
).dropna()

In [ ]:
grid = gpd.read_file(grid_path)

lat_df = grid.assign(lat=grid.geometry.centroid.y)[["GRID_ID", "lat"]]

model_df = model_df.merge(lat_df, on="GRID_ID", how="left")

model_df["insolation_index"] = np.where(
    model_df["lat"] >= 0,
    -model_df["rmax"],
    model_df["rmax"]
)

In [ ]:
model_df.to_csv(output_path, index=False, encoding="utf-8-sig")